In [0]:
#-------------------------------------------------------------------------------------------#
# Data criacao: 15/11/2025                                                                  #
# Objetivo:                                                                                 #
#   Criar uma tabela de produtos já unindo categoria, marca (brands) e quantidade total     #   #   de estoque em todas as lojas                                                            #  
#    Origem camada bronze: brands                                                           #
#                          categories                                                       #
#                          products                                                         #
#                          stocks                                                           #
#    Destino camada silver: product                                                         #
#-------------------------------------------------------------------------------------------#
# 1. Criar uma tabela temporária com os dados da tabela bronze                              #
# 2. Criar uma tabela Silver com os dados da tabela bronze                                  #
# 4. Importar as funções desejadas                                                          #
#-------------------------------------------------------------------------------------------#

In [0]:
import importlib
import os
import sys

# Sobe 2 níveis (de src/02.silver ou src/03.gold -> src -> raiz onde está funcoes_diversas)
caminho_codigo = os.path.abspath("../..")

if caminho_codigo not in sys.path:
    sys.path.insert(0, caminho_codigo)

import funcoes_diversas.funcoes_util
importlib.reload(funcoes_diversas.funcoes_util)

from funcoes_diversas.funcoes_util import buscar_caminho, gerar_tabela_temporaria



In [0]:
mapa_criado = gerar_tabela_temporaria(["brands", "categories", "products","stocks"])

In [0]:
df_product_silver = spark.sql("""with stocks as(
    select product_id, 
       sum(quantity) as total_stocks 
       from tmp_bronze_stocks 
       group by product_id)
    select p.product_id,
           p.product_name,
           b.brand_name,
           c.category_name,
           p.model_year,
           p.list_price,
           s.total_stocks
    from tmp_bronze_products        as p
    left join tmp_bronze_categories as c on p.category_id = c.category_id
    left join tmp_bronze_brands     as b on p.brand_id    = b.brand_id
    left join stocks                as s on p.product_id  = s.product_id """)

In [0]:
silver_path = buscar_caminho("silver")
# transformando consulta em uma tabela Silver 
df_product_silver.write\
    .mode("overwrite")\
    .format("delta")\
    .option("overwriteSchema", "true")\
    .save(f"{silver_path}/product")